In [ ]:
"""
01_rag_pipeline.py
V0 (순수 LLM) → V1 (RAG) → V2 (조문 참조 그래프) 파이프라인
- 임베딩: sentence-transformers (로컬)
- 채팅:   Groq API (무료, llama-3.3-70b)
"""

import os
import json
import numpy as np
from pathlib import Path
from groq import Groq
from sentence_transformers import SentenceTransformer

# ──────────────────────────────────────────
# 설정
# ──────────────────────────────────────────
GROQ_API_KEY = ""   
# console.groq.com에서 발급
client     = Groq(api_key=GROQ_API_KEY)
CHAT_MODEL = "llama-3.3-70b-versatile"
TOP_K      = 5
REF_DEPTH  = 1

CHUNKS_FILE = Path("data/chunks.json")
DB_FILE     = Path("data/vectordb.json")

print("임베딩 모델 로딩 중…")
_embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("임베딩 모델 로딩 완료")


# ──────────────────────────────────────────
# 임베딩 (로컬)
# ──────────────────────────────────────────
def embed(texts: list) -> list:
    return _embedder.encode(texts, normalize_embeddings=True).tolist()

def embed_query(text: str) -> list:
    return _embedder.encode(text, normalize_embeddings=True).tolist()


# ──────────────────────────────────────────
# 벡터 DB
# ──────────────────────────────────────────
def cosine_sim(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))


class SimpleVectorDB:
    def __init__(self):
        self.ids      = []
        self.texts    = []
        self.vectors  = []
        self.metadata = []

    def add(self, chunk_id, text, vector, meta):
        self.ids.append(chunk_id)
        self.texts.append(text)
        self.vectors.append(vector)
        self.metadata.append(meta)

    def search(self, query_vector, top_k=TOP_K):
        sims    = [cosine_sim(query_vector, v) for v in self.vectors]
        top_idx = sorted(range(len(sims)), key=lambda i: sims[i], reverse=True)[:top_k]
        return [{**self.metadata[i], "score": sims[i]} for i in top_idx]

    def get_by_id(self, chunk_id):
        for i, cid in enumerate(self.ids):
            if chunk_id == cid or cid.startswith(chunk_id):
                return self.metadata[i]
        return None

    def save(self, path):
        path.write_text(json.dumps({
            "ids": self.ids, "texts": self.texts,
            "vectors": self.vectors, "metadata": self.metadata,
        }, ensure_ascii=False), encoding="utf-8")

    def load(self, path):
        data      = json.loads(path.read_text(encoding="utf-8"))
        self.ids      = data["ids"]
        self.texts    = data["texts"]
        self.vectors  = data["vectors"]
        self.metadata = data["metadata"]


# ──────────────────────────────────────────
# 인덱스 빌드
# ──────────────────────────────────────────
def build_index(chunks_file: Path, db_file: Path):
    data      = json.loads(chunks_file.read_text(encoding="utf-8"))
    chunks    = data["chunks"]
    ref_graph = data["ref_graph"]

    if db_file.exists():
        print("기존 벡터 DB 로드 중...")
        db = SimpleVectorDB()
        db.load(db_file)
        return db, ref_graph

    print(f"{len(chunks)}개 chunk 임베딩 중...")
    db    = SimpleVectorDB()
    BATCH = 100
    for i in range(0, len(chunks), BATCH):
        batch = chunks[i:i+BATCH]
        texts = [f"{c['article_num']} {c.get('article_title','')} {c['text']}" for c in batch]
        vecs  = embed(texts)
        for chunk, vec in zip(batch, vecs):
            db.add(chunk["id"], chunk["text"], vec, chunk)
        print(f"  {min(i+BATCH, len(chunks))}/{len(chunks)}")

    db.save(db_file)
    print(f"벡터 DB 저장: {db_file}")
    return db, ref_graph


# ──────────────────────────────────────────
# 조문 포매터
# ──────────────────────────────────────────
def format_chunks(chunks: list) -> str:
    lines = []
    for c in chunks:
        title = f"({c.get('article_title','')})" if c.get("article_title") else ""
        lines.append(f"[{c['id']}]{title} {c['text']}")
    return "\n\n".join(lines)


# ──────────────────────────────────────────
# LLM 채팅 헬퍼
# ──────────────────────────────────────────
def chat(system_prompt: str, user_prompt: str) -> str:
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
    )
    return resp.choices[0].message.content


# ──────────────────────────────────────────
# V0: 순수 LLM (Baseline)
# ──────────────────────────────────────────
def v0_baseline(question: str) -> dict:
    answer = chat(
        system_prompt=(
            "당신은 한국 AI기본법 전문가입니다. "
            "사전 학습 지식으로만 답변하고, 조문을 인용할 때는 정확한 조문 번호를 명시하세요."
        ),
        user_prompt=question,
    )
    return {"version": "V0", "answer": answer, "retrieved_chunks": []}


# ──────────────────────────────────────────
# V1: 기본 RAG
# ──────────────────────────────────────────
def v1_rag(question: str, db: SimpleVectorDB) -> dict:
    q_vec   = embed_query(question)
    results = db.search(q_vec, top_k=TOP_K)
    context = format_chunks(results)

    answer = chat(
        system_prompt="""당신은 AI기본법 컴플라이언스 QA 에이전트입니다.
규칙:
1. 반드시 아래 [관련 조문]만을 근거로 답변하세요.
2. 각 주장마다 조문 번호를 [제○조 제○항] 형식으로 인용하세요.
3. 조문에 근거가 없으면 "현재 제공된 조문에서 확인할 수 없습니다."라고 명시하세요.
4. 법률 자문이 아닌 문서 기반 QA임을 명심하세요.""",
        user_prompt=f"[관련 조문]\n{context}\n\n[질문]\n{question}",
    )
    return {
        "version":          "V1",
        "answer":           answer,
        "retrieved_chunks": [c["id"] for c in results],
        "scores":           [round(c["score"], 4) for c in results],
    }


# ──────────────────────────────────────────
# V2: RAG + 조문 참조 그래프 탐색
# ──────────────────────────────────────────
def v2_rag_with_refs(question: str, db: SimpleVectorDB, ref_graph: dict, depth: int = REF_DEPTH) -> dict:
    q_vec   = embed_query(question)
    initial = db.search(q_vec, top_k=TOP_K)

    expanded_ids = set(c["id"] for c in initial)
    frontier     = list(expanded_ids)

    for _ in range(depth):
        next_frontier = []
        for cid in frontier:
            chunk_meta = db.get_by_id(cid)
            if chunk_meta:
                for ref in chunk_meta.get("refs", []):
                    ref_chunk = db.get_by_id(ref)
                    if ref_chunk and ref_chunk["id"] not in expanded_ids:
                        expanded_ids.add(ref_chunk["id"])
                        initial.append(ref_chunk)
                        next_frontier.append(ref_chunk["id"])
        frontier = next_frontier

    context = format_chunks(initial)
    answer  = chat(
        system_prompt="""당신은 AI기본법 컴플라이언스 QA 에이전트입니다.
규칙:
1. 반드시 아래 [관련 조문]만을 근거로 답변하세요.
2. 각 주장마다 조문 번호를 [제○조 제○항] 형식으로 인용하세요.
3. 조문에 근거가 없으면 "현재 제공된 조문에서 확인할 수 없습니다."라고 명시하세요.""",
        user_prompt=f"[관련 조문]\n{context}\n\n[질문]\n{question}",
    )
    return {
        "version":          "V2",
        "answer":           answer,
        "retrieved_chunks": list(expanded_ids),
        "initial_chunks":   [c["id"] for c in initial[:TOP_K]],
        "expanded_chunks":  list(expanded_ids - set(c["id"] for c in initial[:TOP_K])),
    }


# ──────────────────────────────────────────
# 실행 예시
# ──────────────────────────────────────────
if __name__ == "__main__":
    db, ref_graph = build_index(CHUNKS_FILE, DB_FILE)

    test_questions = [
        "저희는 AI로 채용 서류를 1차 스크리닝하는데, 최종 합격은 사람이 결정합니다. 고영향 AI인가요?",
        "AI 서비스 출시 전에 반드시 해야 하는 의무 고지 사항은 무엇인가요?",
        "AI기본법에서 '사업자'의 정의는 무엇인가요?",
        "AI 생성 이미지에 워터마크를 반드시 붙여야 하나요?",
    ]

    for q in test_questions:
        print(f"\n{'='*60}")
        print(f"[질문] {q}")

        r0 = v0_baseline(q)
        print(f"\n[V0 - Baseline]\n{r0['answer'][:300]}…")

        r1 = v1_rag(q, db)
        print(f"\n[V1 - RAG] 검색된 조문: {r1['retrieved_chunks']}")
        print(r1['answer'][:300], "…")

        r2 = v2_rag_with_refs(q, db, ref_graph)
        print(f"\n[V2 - RAG+Graph] 확장 조문: {r2['expanded_chunks']}")
        print(r2['answer'][:300], "…")
""

임베딩 모델 로딩 중…


model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

임베딩 모델 로딩 완료
476개 chunk 임베딩 중...
  100/476
  200/476
  300/476
  400/476
  476/476
벡터 DB 저장: data/vectordb.json

[질문] 저희는 AI로 채용 서류를 1차 스크리닝하는데, 최종 합격은 사람이 결정합니다. 고영향 AI인가요?

[V0 - Baseline]
한국 AI기본법 제2조 제1항에서는 고영향 AI를 다음과 같이 정의하고 있습니다.

"제2조(정의) ① 이 법에서 사용하는 용어의 뜻은 다음과 같다.
1. '인공지능'이란 인간이 설계·개발한 시스템이 자율적으로 지능적 행위를 하는 것을 말한다.
2. '고영향 인공지능'이란 영향평가 결과 고영향으로 판정된 인공지능을 말한다."

다만, 고영향 AI의 구체적인 판정 기준은 별도로 명시되어 있지 않습니다. 다만, 같은 법 제24조에서 "고영향 판정의 기준에 관한령 제17조에서" 고영향 AI의 판정 기준에 대해 별도의 기준을 두고 있습니다…

[V1 - RAG] 검색된 조문: ['제10조제2항음호', '제35조', '제28조제4항다호', '제10조제2항임호', '제33조제4항']
[제10조제2항]에 따르면, AI 시스템의 분석 과정이 불투명하여 후보자가 본인의 평가 결과 산출 근거를 확인하거나 이의제기하기 어려워 알 권리가 제한될 가능성이 있음을 확인할 수 있습니다. 또한, [제33조제4항]에 따르면 고영향 인공지능의 확인에 필요한 사항은 대통령령으로 정합니다. 하지만 제공된 정보만으로는 고영향 인공지능인지 여부를確定적으로 판단할 수 없습니다. 고영향 인공지능 여부를 확인하려면 [제33조제4항]에서 언급된 대통령령에서 정한 기준을 참조해야 할 것입니다. 현재 제공된 조문에서 확인할 수 없습니다. …

[V2 - RAG+Graph] 확장 조문: ['제25조']
제10조 제2항 음호에 따르면, AI 시스템의 분석 과정이 불투명하여 후보자가 본인의 평가 결과 산출 근거를 확인하거나 이의제기하기 어려워 알 권리가 제한될 가능성이 있는 경우

''